# chunker.py

In [ ]:
import re
from typing import List

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

def build_splitter(config) -> RecursiveCharacterTextSplitter:
  """Builds a text splitter .
  Priority
  1. Dieu
  2. Khoan
  3. Diem
  4. Paragraph
  5. Line
  6. Word
  """
  return RecursiveCharacterTextSplitter(
    separators=[
      "\nĐiều ",
      "\nKhoản ",
      "\nĐiểm ",
      "\nMục ",
      "\nChương ",
      "\n\n",
      "\n",
      " ",
      "",
    ],
    chunk_size=config["chunking"]["chunk_size"],
    chunk_overlap=config["chunking"]["chunk_overlap"],
  )

def extract_article_number(text: str):
  """
  Try extracting article number from the text.
  Example: "Điều 1. Quyền và nghĩa vụ của công dân" -> "1"
  """
  match = re.search(r"Điều\s+(\d+)", text)
  if match:
      return match.group(1)
  return None

def chunk_documents(docs: List[Document], config) -> List[Document]:
  """Split cleaned documents into legal chunks."""

  splitter = build_splitter(config)

  chunks = splitter.split_documents(docs)

  for i, chunk in enumerate(chunks):
    #  Track chunk order
    chunk.metadata["chunk_index"] = i

    # Kepp chunk size for debugging
    chunk.metadata["chunk_length"] = len(chunk.page_content)

    # Save article number if found
    article_number = extract_article_number(chunk.page_content)
    if article_number:
      chunk.metadata["article_number"] = article_number
  
  return chunks



# cleaner.py

In [ ]:
import os
import re
import unicodedata
from typing import List
from concurrent.futures import ProcessPoolExecutor

from bs4 import BeautifulSoup
from tqdm import tqdm
from langchain_core.documents import Document



def _strip_html(html: str) -> str:
  """
  Remove HTML tags while preserving readable text structure.

  Tai sao nen dung BeautifulSoup thay vi regex de strip HTML:
  - an toan cho nested tags
  - dam bao cau truc van ban duoc giu nguyen  
  """
  soup = BeautifulSoup(html, "lxml")

  # Chuyen block tags thanh newline de giu cau truc van ban
  text = soup.get_text(separator="\n")

  return text

def _normalize(text: str) -> str:
  """
  Normalize text gon gang de co the tim kiem de dang
  Steps:
  - Unicode NFC normalization (tuc la chuyen cac ky tu ve dang chuan)
  - Remove excessive whitespace (tuc la thay nhieu khoang trang lien tiep bang 1 khoang trang duy nhat)
  - Collapse multiple empty lines (tuc la thay nhieu dong trong lien tiep bang 1 dong trong duy nhat)
  """

  # Normalize unicode (quan trong cho tieng Viet)
  text = unicodedata.normalize("NFC", text)

  # Remove non-breaking spaces
  text = text.replace("\xa0", " ")

  # Remove excessive spaces/tabs
  text = re.sub(r"[ \t]+", " ", text)

  # Collapse too many newlines into max 2
  text = re.sub(r"\n{3,}", "\n\n", text)

  # Strip leading/trailing spaces
  text = text.strip()

  return text

def clean_document(doc: Document) -> Document:
  """
  Clean a single document.

  Pipeline:
  raw HTML -> strip HTML -> normalize unicode -> clean whitespace
  """
  raw = doc.page_content
  # Only parse HTML if document looks like HTML
  if "<" in raw and ">" in raw:
    raw = _strip_html(raw)

  cleaned = _normalize(raw)
  return Document(page_content=cleaned, metadata=doc.metadata)


def clean_documents(docs: List[Document], workers: int = 1) -> List[Document]:
  """
  Clean multiple documents

  workers = 1: -> sequential processing

  workers > 1: -> parallel processing using ProcessPoolExecutor

  """
  
  # Sequential mode (easy to debug)
  if workers <= 1:
    return [
      clean_document(doc) for doc in tqdm(docs, desc="Cleaning documents", unit="doc")
    ]
  
  # Parallel mode (faster for large datasets)
  with ProcessPoolExecutor(max_workers=workers) as executor:
    cleaned_docs = list(
      tqdm(
        executor.map(
          clean_document,
          docs,
          chunksize=64
        ),
        total=len(docs),
        desc="Cleaning documents",
        unit="doc"
      )
    )
  return cleaned_docs



# loader

In [ ]:


import pandas as pd
from typing import List, Optional
from datasets import load_dataset
from langchain_core.documents import Document

def load_documents(
    config: dict,
    sample_size: Optional[int] = None,
) -> List[Document]:
    """
    Tải tài liệu bằng chiến lược kết hợp: 
    HF Datasets (Metadata) + Pandas (Content Parquet) để né lỗi PyArrow.
    """
    dataset_name = config["dataset"]["name"]

    print("--> 1. Đang tải metadata (Hugging Face Datasets)...")
    metadata_ds = load_dataset(dataset_name, "metadata", split="data")
    
    # Ép doc_id về dạng string để đảm bảo map chính xác
    metadata_lookup = {}
    for row in metadata_ds:
        doc_id = row.get("id")
        if doc_id is not None:
            metadata_lookup[str(doc_id)] = row

    print("--> 2. Đang tải nội dung trực tiếp bằng Pandas...")
    # Đường dẫn file Parquet gốc trên Hugging Face Hub
    parquet_url = f"hf://datasets/{dataset_name}/data/content.parquet"
    
    # Pandas đọc trực tiếp file parquet sẽ không bị lỗi ép kiểu 32-bit
    content_df = pd.read_parquet(parquet_url)

    if sample_size:
        content_df = content_df.sample(n=sample_size, random_state=42)

    docs = []
    skipped = 0

    print("--> 3. Đang xử lý tài liệu và chuyển thành LangChain Documents...")
    for _, row in content_df.iterrows():
        doc_id = row.get("id")
        content_html = row.get("content_html")

        # Kiểm tra None hoặc NaN trong Pandas
        if pd.isna(doc_id) or pd.isna(content_html) or not str(content_html).strip():
            skipped += 1
            continue

        doc_id_str = str(doc_id)
        meta = metadata_lookup.get(doc_id_str, {})

        docs.append(
            Document(
                page_content=str(content_html),
                metadata={
                    "doc_id": doc_id_str,
                    "title": meta.get("title", ""),
                    "doc_type": meta.get("loai_van_ban", ""),
                    "authority": meta.get("co_quan_ban_hanh", ""),
                    "issue_date": meta.get("ngay_ban_hanh", ""),
                    "effective_date": meta.get("ngay_co_hieu_luc", ""),
                    "status": meta.get("tinh_trang_hieu_luc", ""),
                }
            )
        )
        
    print(f"[Thành công] Đã tải {len(docs)} tài liệu.")
    print(f"[Bỏ qua] Đã bỏ qua {skipped} dòng không hợp lệ.")

    return docs



def load_relationships(
    config: dict,
    sample_size: Optional[int] = None,
) -> list[dict]:
    """
    Load quan hệ giữa các văn bản pháp luật.
    """

    dataset_name = config["dataset"]["name"]

    print("--> Đang tải relationships...")

    relationship_ds = load_dataset(
        dataset_name,
        "relationships",
        split="data"
    )

    if sample_size:
        relationship_ds = relationship_ds.select(
            range(sample_size)
        )

    relationships = []

    for row in relationship_ds:
        doc_id = row.get("doc_id")
        other_doc_id = row.get("other_doc_id")
        relationship = row.get("relationship")

        if not doc_id or not other_doc_id:
            continue

        relationships.append({
            "doc_id": str(doc_id),
            "other_doc_id": str(other_doc_id),
            "relationship": relationship
        })

    print(f"[Thành công] Loaded {len(relationships)} relationships")

    return relationships




# bm25_index.py

In [ ]:
from __future__ import annotations

# pickle de save/load bm25 object xuong disk
import os
import pickle

from langchain_core.documents import Document
from rank_bm25 import BM25Okapi

class BM25Index:
  def __init__(
    self,
    persist_path: str = "data/bm25_index.pkl",
  ):
    self.persist_path = persist_path
    self.bm25 = None
    self.documents = []
  
  def tokenize(self, text: str) -> list[str]:
    # Simple tokenizer: lowercase + split by space
    # Vi du:
    """ Điều 1 quy định chung ->  ["điều", "1", "quy", "định", "chung"] """
    return text.lower().split()
  
  def build(
      self,
      documents: list[Document],
  ):
    # luu full documents de mapping ket qua sau search
    self.documents = documents

    # tokenize tung document
    tokenized_docs = [
      self.tokenize(doc.page_content) for doc in documents
    ]

    # build bm25 index
    self.bm25 = BM25Okapi(tokenized_docs)

    return self.bm25
  
  def save(self):
    if self.bm25 is None:
      raise ValueError("BM25 index is not built yet. Call build() first.")

    os.makedirs(os.path.dirname(self.persist_path), exist_ok=True)
    
    with open(self.persist_path, "wb") as f:
      pickle.dump(
        {
          "bm25": self.bm25,
          "documents": self.documents,
        }, f)
  
  def load(self):
    with open(self.persist_path, "rb") as f:
      data = pickle.load(f)
      self.bm25 = data["bm25"]
      self.documents = data["documents"]
  
  def search(
    self,
    query: str,
    k: int = 5,
  ) -> list[Document]:
    if self.bm25 is None:
      raise ValueError("BM25 index is not built yet. Call build() or load() first.")
    
    tokenized_query = self.tokenize(query)
    scores = self.bm25.get_scores(tokenized_query)

    # Sort theo score giam dan
    ranked_indices = sorted(
      range(len(scores)),
      key=lambda i: scores[i],
      reverse=True,
    )

    # Lay top k documents
    top_docs = [
      self.documents[i] for i in ranked_indices[:k]
    ]
    return top_docs

  
    


# chroma_store.py

In [ ]:
from __future__ import annotations
import os
import shutil

from langchain_core.documents import Document
from langchain_chroma import Chroma
from src.indexing.embeddings import Embedder

class ChromaStore:
  def __init__(
    self,
    persist_directory: str = "data/chroma_store",
  ):
    
    self.persist_directory = persist_directory

    self.embedder = Embedder()

    self.store = None

  def build(
    self,
    documents: list[Document],
  ):
    
    if os.path.exists(self.persist_directory):
      print(f"Removing existing Chroma store at {self.persist_directory}...")
      shutil.rmtree(self.persist_directory)
      
    self.store = Chroma.from_documents(
      documents=documents,
      embedding=self.embedder.langchain_embedding,
      persist_directory=self.persist_directory,
    )
    return self.store
  
  def add_documents(
    self,
    documents: list[Document],
  ):
    if self.store is None:
      raise ValueError("Store is not built yet. Call build() first.")
    
    self.store.add_documents(documents)

  def load(self):
    self.store = Chroma(
      persist_directory=self.persist_directory,
      embedding_function=self.embedder.langchain_embedding,
    )
    return self.store
  
  def similarity_search(
    self,
    query: str,
    k: int = 4,
    filter: dict | None = None,
  ):
    if self.store is None:
      raise ValueError("Store is not built yet. Call build() or load() first.")
    
    return self.store.similarity_search(
      query=query,
      k=k,
      filter=filter,
    )
  


# embedding.py

In [ ]:
from __future__ import annotations

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

# Model dung de convert tu text sang vector embeddings
from sentence_transformers import SentenceTransformer


class Embedder:
  """
  Embedding text

  - Load model embedding
  - Embed docs
  - Embed query

  Class nay ko luu vector, no chi tao vector
  """
  def __init__(
    self,
    model_name: str = "intfloat/multilingual-e5-base",
  ):
    self.model_name = model_name

    # Load model vam ram, nen chi load 1 lan
    self.model = SentenceTransformer(model_name)

    self.langchain_embedding = HuggingFaceEmbeddings(model_name=model_name)
  
  def embed_documents(
    self,
    documents: list[Document],
  ) -> list[list[float]]:
    """
    Embed documents

    Args:
      documents: list of Document

    Returns:
      list of vector embeddings
    """

    texts = [
      doc.page_content for doc in documents
    ]
    # normalize_embeddings=True: chuan hoa vector ve khoang -1 den 1, giup cosine similarity hieu qua hon
    embeddings = self.model.encode(
      texts,
      convert_to_numpy=True,
      normalize_embeddings=True,)
    
    # Convert numpy array to list
    # De serialize, debug, compare
    return embeddings.tolist()
  
  def embed_query(
    self,
    query: str,
  ) -> list[float]:
    """
    Embed query

    Args:
      query: string

    Returns:
      vector embedding
    """
    query_text = f"query: {query}"

    embedding = self.model.encode(
      query_text,
      convert_to_numpy=True,
      normalize_embeddings=True,
    )
    return embedding.tolist()
